## Explore Dataset

In [ ]:
import pandas as pd
import requests as rq
import numpy as np
import geopandas as gpd
import json
import matplotlib.pyplot as plt
import shapely
import dotenv
import os

In [ ]:
dotenv.load_dotenv()
RIVERS_URL = os.getenv("RIVERS_URL")

## Obtaining data

### CSO Data

In [ ]:
geo_JSON_url = "https://services2.arcgis.com/g6o32ZDQ33GpCIu3/arcgis/rest/services/Thames_Water_Storm_Overflow_Activity_(Production)_view/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson"
cso = gpd.read_file(geo_JSON_url)

In [ ]:
cso.geometry

In [ ]:
cso.head()

### River data

In [ ]:
rivers = gpd.read_file(RIVERS_URL)
rivers = rivers.to_crs("EPSG:4326") # Change to lat/long

## Start Plotting geometry

### CSO Data

In [ ]:
cso.explore()

### River Data

In [ ]:
m = rivers[rivers.name1.str.contains("Thames",case=False,na=False)].explore()
cso.explore(m=m)

## Thames analysis

The idea is to try to find CSOs near the river thames and correctly associate them to the river. Then, it would be to assign a simple risk to the river based on the distance of the point from the CSO

### Analysis near the thames river

In [ ]:
cso_thames = cso.copy()
cso_thames["isClose"] = False

In [ ]:
river_thames = rivers[rivers["name1"] == "River Thames"].copy()
river_thames = river_thames.reset_index().drop(columns="index")

In [ ]:
# Dividing each river segment into equally spaced points

n_divisions = 10
river_thames["geometry2"] = river_thames.geometry.apply(lambda x: shapely.line_interpolate_point(x,np.linspace(0,1,n_divisions),normalized=True))

In [ ]:
river_thames_expl = river_thames.explode("geometry2")

In [ ]:
river_thames_expl.geometry2 = river_thames_expl.geometry2.astype("geometry")

In [ ]:
river_thames_expl.geometry2.set_crs(river_thames_expl.geometry.crs)

In [ ]:
river_thames_points = river_thames_expl.set_geometry("geometry2")

In [ ]:
river_thames_points

In [ ]:
river_thames_points[river_thames_points["OBJECTID"]==135003]

In [ ]:
m = river_thames_points[river_thames_points["OBJECTID"]==135003].explore()
river_thames[river_thames["OBJECTID"]==135003].explore(m=m)

#### METHOD 1 : Using buffer distance

In [ ]:
bounds_distance = 100 # m
river_thames_bounds = river_thames.copy()
river_thames_bounds["bounds"] = river_thames.geometry.apply(lambda x: )